# Scratch

## iseg sanity test

In [ ]:
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd().resolve()
if not (NOTEBOOK_DIR / "scratch.ipynb").exists():
    NOTEBOOK_DIR = (Path.cwd() / "notebooks" / "full_model").resolve()
LFM_ROOT = NOTEBOOK_DIR.parents[1]
GRAHA_ROOT = LFM_ROOT / "lfm" / "full_model" / "graha-lunar-fm"

PRETRAIN_DIR = Path(
    "/explore/nobackup/projects/lfm/gabby/Lunar-FM/experiments/"
    "lunarfm_base_dual_full_nas_no_nans_256_256_lr1e-4_wd0.05"
).resolve()
BACKBONE_WEIGHTS = PRETRAIN_DIR / "checkpoints/checkpoint_weights_final.pt"
BACKBONE_CFG = PRETRAIN_DIR / "full_config.yaml"
MODALITY_INFO = PRETRAIN_DIR / "modality_info.yaml"
NORMALIZED_WAC_DATA_RANGE = [-1.0, 1.0]

for import_path in [GRAHA_ROOT, LFM_ROOT]:
    if str(import_path) not in sys.path:
        sys.path.insert(0, str(import_path))

from lfm.full_model.datamodules.instance_segmentation import (
    LunarInstanceSegmentationDatamodule,
    LunarObjectDetectionInstanceSegmentationDatamodule,
)
from lfm.full_model.utils import create_timestamped_output_dir, plot_instance_batch_sanity
from lfm.full_model.utils.utils import ensure_data_symlink

print("Notebook directory:", NOTEBOOK_DIR)
print("LFM root:", LFM_ROOT)
print("Graha/Lunar-FM code root:", GRAHA_ROOT)
print("Backbone weights:", BACKBONE_WEIGHTS)

In [ ]:
ISEG_DATA_ROOT = Path("/panfs/ccds02/nobackup/projects/lfm/model_inputs/300_300_inputs/full_model_inst_seg_v2")  # expects train/val/test/{chips,labels}
ISEG_OUTPUT_DIR = create_timestamped_output_dir(NOTEBOOK_DIR / "outputs" / "instance_sanity")

iseg_datamodule = LunarInstanceSegmentationDatamodule(
    data_root=ISEG_DATA_ROOT,
    crop_size=256,
    batch_size=5,
    num_workers=0,
)

plot_instance_batch_sanity(
    iseg_datamodule,
    output_dir=ISEG_OUTPUT_DIR,
    split="train",
    n_samples=5,
)

### ObjectDetectionTask target sanity test

This checks the true instance target format expected by TerraTorch `ObjectDetectionTask`: `image`, `boxes`, `labels`, and `masks`.

In [ ]:
OD_ISEG_DATA_ROOT = ISEG_DATA_ROOT

od_iseg_datamodule = LunarObjectDetectionInstanceSegmentationDatamodule(
    data_root=OD_ISEG_DATA_ROOT,
    crop_size=256,
    batch_size=4,
    num_workers=0,
    target_box_format="xyxy",  # pixel xyxy for mask-rcnn
)
od_iseg_datamodule.setup("fit")
od_batch = next(iter(od_iseg_datamodule.train_dataloader()))

print("batch keys:", od_batch.keys())
print("image:", tuple(od_batch["image"].shape), od_batch["image"].dtype)
print("boxes per image:", [tuple(x.shape) for x in od_batch["boxes"]])
print("labels per image:", [tuple(x.shape) for x in od_batch["labels"]])
print("masks per image:", [tuple(x.shape) for x in od_batch["masks"]])
print("first filename:", od_batch["filename"][0])

for i, (boxes, labels, masks) in enumerate(zip(od_batch["boxes"], od_batch["labels"], od_batch["masks"])):
    assert boxes.ndim == 2 and boxes.shape[-1] == 4
    assert labels.ndim == 1
    assert masks.ndim == 3 and masks.shape[-2:] == od_batch["image"].shape[-2:]
    assert boxes.shape[0] == labels.shape[0] == masks.shape[0]
    if boxes.numel():
        assert bool((boxes[:, 2] > boxes[:, 0]).all())
        assert bool((boxes[:, 3] > boxes[:, 1]).all())
print("ObjectDetectionTask target sanity check passed.")

### Graha Mask R-CNN smoke test

This builds the first true instance-segmentation path: WAC inputs, object-detection targets, TerraTorch `ObjectDetectionTask`, and `framework="mask-rcnn"`. Run the target sanity cell above first.

In [ ]:
import torch
from lightning.pytorch import Trainer, seed_everything

# Importing terratorch_integration registers lunarmind_v1_* backbones, custom necks,
# and LunarObjectDetectionTask.
import terratorch_integration  # noqa: F401
from terratorch_integration.lunar_object_detection_task import LunarObjectDetectionTask

seed_everything(42)

required_paths = [GRAHA_ROOT, BACKBONE_WEIGHTS, BACKBONE_CFG, MODALITY_INFO, OD_ISEG_DATA_ROOT]
missing = [path for path in required_paths if not Path(path).exists()]
if missing:
    raise FileNotFoundError("Missing required paths:\n" + "\n".join(str(path) for path in missing))

In [ ]:
od_iseg_datamodule = LunarObjectDetectionInstanceSegmentationDatamodule(
    data_root=OD_ISEG_DATA_ROOT,
    crop_size=256,
    batch_size=2,
    num_workers=0,
    target_box_format="xyxy",
)
od_iseg_datamodule.setup("fit")
sample_batch = next(iter(od_iseg_datamodule.train_dataloader()))

WAC_MODALITY = "wac"
WAC_NUM_CHANNELS = int(sample_batch["image"].shape[1])

print("sample image:", tuple(sample_batch["image"].shape), sample_batch["image"].dtype)
print("target boxes per image:", [tuple(x.shape) for x in sample_batch["boxes"]])
print("target masks per image:", [tuple(x.shape) for x in sample_batch["masks"]])
print("WAC channels registered for model:", WAC_NUM_CHANNELS)

od_task = LunarObjectDetectionTask(
    model_factory="ObjectDetectionModelFactory",
    model_args={
        "framework": "mask-rcnn",
        "backbone": "lunarmind_v1_base",
        "backbone_checkpoint_path": str(BACKBONE_WEIGHTS),
        "backbone_cfg": str(BACKBONE_CFG),
        "backbone_modality_info_path": str(MODALITY_INFO),
        "backbone_modalities": [WAC_MODALITY],
        "backbone_new_modalities": {
            WAC_MODALITY: {
                "type": "image",
                "num_channels": WAC_NUM_CHANNELS,
                "data_range": NORMALIZED_WAC_DATA_RANGE,
            },
        },
        "num_classes": 2,
        "in_channels": WAC_NUM_CHANNELS,
        "framework_min_size": 256,
        "framework_max_size": 256,
        "backbone_patch_size": 8,
        "backbone_remove_register_tokens": False,
        "backbone_merge_method": None,
        "necks": [
            {"name": "SelectIndices", "indices": [2, 5, 8, 11]},
            {"name": "ReshapeTokensToImage", "remove_cls_token": False, "h": 32},
            {"name": "LearnedInterpolateToPyramidal"},
            {"name": "FeaturePyramidNetworkNeck"},
        ],
    },
    freeze_backbone=False,
    freeze_decoder=False,
    class_names=["Background", "Crater"],
    backbone_lr=5.0e-5,
    head_lr=2.0e-4,
    layer_decay=0.75,
    weight_decay=0.05,
    warmup_steps=0,
    anchor_sizes=[[4], [8], [16], [32], [64]],
    anchor_aspect_ratios=[0.5, 1.0, 2.0],
    score_threshold=0.5,
)

print(type(od_task.model))

In [ ]:
# Fastest smoke check: one direct Mask R-CNN loss call, without attaching a Trainer.
# This verifies that the task can reformat the batch and the model returns instance losses.
od_task.train()
x = sample_batch["image"]
targets = od_task.reformat_batch(sample_batch, batch_size=x.shape[0])
loss_dict = od_task(x, targets)
if not isinstance(loss_dict, dict):
    loss_dict = loss_dict.output
loss = sum(loss_dict.values())
print("loss terms:", {key: float(value.detach().cpu()) for key, value in loss_dict.items()})
print("single-step train loss:", float(loss.detach().cpu()))

In [ ]:
RUN_LIGHTNING_OD_SMOKE = False

if RUN_LIGHTNING_OD_SMOKE:
    trainer = Trainer(
        accelerator="gpu" if torch.cuda.is_available() else "cpu",
        devices=1,
        precision="32",
        max_epochs=1,
        fast_dev_run=2,
        logger=False,
        enable_checkpointing=False,
    )
    trainer.fit(od_task, datamodule=od_iseg_datamodule)

In [ ]:
a

## Examine data

In [ ]:
from pathlib import Path

LFM_ROOT = Path("/panfs/ccds02/nobackup/projects/lfm/")
DATA_ROOT = LFM_ROOT / "model_inputs/300_300_inputs/kaguya_static_all_wac"
ISEG = DATA_ROOT / "inst_seg"
ISEG_LABELS = ISEG / "labels"

label_files = list(ISEG_LABELS.glob("*.npz"))
example_label = label_files[0]
example_label_archive = np.load(example_label, allow_pickle=True)
label = example_label_archive
print(f"example label loaded: {example_label_archive}")
example_data = example_label_archive['mask']
example_bboxes = example_label_archive['bboxes']

print(example_data.shape, "\n", example_bboxes)

In [ ]:
example_n_craters = example_label_archive['num_craters']
print(f"num_craters: {example_n_craters}, len(bboxes): {len(example_bboxes)}")

print(f"Unique mask values: {np.unique(label["mask"])}")

print('Datatypes of mask, bboxes, num_craters: ')
print(label["mask"].dtype)
print(label["bboxes"].dtype)
print(label["num_craters"].dtype)

## Create 7 band dataset
Creates `data_7band/{split}/{chips,labels}` from `data/{split}/{chips,labels}`. Chip TIFFs keep only bands 1-7. Labels are copied unchanged.

In [ ]:
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path
import logging
import os
import shutil

try:
    import pyproj
    os.environ["PROJ_LIB"] = pyproj.datadir.get_data_dir()
except Exception:
    pass

logging.getLogger("rasterio._env").setLevel(logging.ERROR)

import rasterio

SRC_ROOT = Path("data")
DST_ROOT = Path("data_7band")
MAX_WORKERS = 16


def write_7band_chip(args):
    chip_path, out_path = args
    out_path.parent.mkdir(parents=True, exist_ok=True)

    with rasterio.open(chip_path) as src:
        if src.count < 7:
            raise ValueError(f"{chip_path} has only {src.count} bands")

        profile = src.profile.copy()
        profile.update(driver="GTiff", count=7)
        data = src.read(indexes=list(range(1, 8)))

        with rasterio.open(out_path, "w", **profile) as dst:
            dst.write(data)

    return str(out_path)


def copy_label(args):
    label_path, out_path = args
    out_path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(label_path, out_path)
    return str(out_path)


def process_split(split):
    src_chips = SRC_ROOT / split / "chips"
    src_labels = SRC_ROOT / split / "labels"
    dst_chips = DST_ROOT / split / "chips"
    dst_labels = DST_ROOT / split / "labels"

    chip_jobs = [(p, dst_chips / p.name) for p in sorted(src_chips.glob("*.tif"))]
    label_jobs = [(p, dst_labels / p.name) for p in sorted(src_labels.iterdir()) if p.is_file()]

    with ProcessPoolExecutor(max_workers=MAX_WORKERS) as executor:
        chip_outputs = list(executor.map(write_7band_chip, chip_jobs))

    with ProcessPoolExecutor(max_workers=MAX_WORKERS) as executor:
        label_outputs = list(executor.map(copy_label, label_jobs))

    print(f"{split}: wrote {len(chip_outputs)} chips and copied {len(label_outputs)} labels")

In [ ]:
# for split in ["train"]:
#     process_split(split)

In [ ]:
# for split in ["val"]:
#     process_split(split)

In [ ]:
# for split in ["test"]:
#     process_split(split)

## Create Semantic Segmentation Splits

Builds a fresh `train`/`val`/`test` split from `kaguya_static_all_wac/sem_seg`. Chip TIFFs are saved with only the first 7 bands. Labels are copied unchanged; the label-processing helper is kept as a no-op hook.

In [ ]:
import random
import shutil
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path

import rasterio

LFM_ROOT = Path("/explore/nobackup/projects/lfm/model_inputs/300_300_inputs")
SEM_SEG_ROOT = LFM_ROOT / "kaguya_static_all_wac" / "sem_seg"
SEM_SEG_OUTPUT_ROOT = LFM_ROOT / "full_model_sem_seg_v2"

SEM_SEG_CHIPS_DIR = SEM_SEG_ROOT / "chips"
SEM_SEG_LABELS_DIR = SEM_SEG_ROOT / "labels"

SEM_SEG_IMAGE_GLOB = "*.tif"
SEM_SEG_LABEL_GLOB = "*_label.*"
SEM_SEG_IMAGE_SUFFIX = "_input_wac_static_chip"
SEM_SEG_LABEL_SUFFIX = "_label"

SEM_SEG_SEED = 42
SEM_SEG_N_TEST = 100
SEM_SEG_TRAIN_FRACTION = 0.95
SEM_SEG_MAX_WORKERS = 16


def split_key(path, suffix):
    stem = path.stem
    if suffix and stem.endswith(suffix):
        return stem[: -len(suffix)]
    return stem


def find_sem_seg_pairs():
    chips = {
        split_key(path, SEM_SEG_IMAGE_SUFFIX): path
        for path in sorted(SEM_SEG_CHIPS_DIR.glob(SEM_SEG_IMAGE_GLOB))
    }
    labels = {
        split_key(path, SEM_SEG_LABEL_SUFFIX): path
        for path in sorted(SEM_SEG_LABELS_DIR.glob(SEM_SEG_LABEL_GLOB))
    }

    keys = sorted(set(chips) & set(labels))
    chip_only = sorted(set(chips) - set(labels))
    label_only = sorted(set(labels) - set(chips))

    print(f"matched pairs: {len(keys)}")
    print(f"chips only:    {len(chip_only)}")
    print(f"labels only:   {len(label_only)}")

    if len(keys) <= SEM_SEG_N_TEST:
        raise ValueError(f"Need more than {SEM_SEG_N_TEST} pairs, found {len(keys)}")

    return [(chips[key], labels[key]) for key in keys]


def process_sem_seg_label(label_path: Path, label_out: Path) -> None:
    label_out.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(label_path, label_out)


def write_sem_seg_7band_chip(chip_path: Path, out_path: Path) -> None:
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with rasterio.open(chip_path) as src:
        if src.count < 7:
            raise ValueError(f"{chip_path} has only {src.count} bands")
        profile = src.profile.copy()
        profile.update(driver="GTiff", count=7)
        data = src.read(indexes=list(range(1, 8)))
        with rasterio.open(out_path, "w", **profile) as dst:
            dst.write(data)


def copy_sem_seg_pair_job(args):
    chip_path, label_path, split = args
    chip_out = SEM_SEG_OUTPUT_ROOT / split / "chips" / chip_path.name
    label_out = SEM_SEG_OUTPUT_ROOT / split / "labels" / label_path.name
    write_sem_seg_7band_chip(chip_path, chip_out)
    process_sem_seg_label(label_path, label_out)
    return split


def create_sem_seg_split():
    pairs = find_sem_seg_pairs()
    rng = random.Random(SEM_SEG_SEED)
    rng.shuffle(pairs)

    test_pairs = pairs[:SEM_SEG_N_TEST]
    remaining = pairs[SEM_SEG_N_TEST:]
    n_train = int(round(len(remaining) * SEM_SEG_TRAIN_FRACTION))

    split_pairs = {
        "train": remaining[:n_train],
        "val": remaining[n_train:],
        "test": test_pairs,
    }

    jobs = [
        (chip_path, label_path, split)
        for split, pairs_for_split in split_pairs.items()
        for chip_path, label_path in pairs_for_split
    ]

    with ProcessPoolExecutor(max_workers=SEM_SEG_MAX_WORKERS) as executor:
        list(executor.map(copy_sem_seg_pair_job, jobs))

    for split, pairs_for_split in split_pairs.items():
        print(f"{split}: {len(pairs_for_split)} pairs")
    print(f"wrote split dataset to: {SEM_SEG_OUTPUT_ROOT.resolve()}")

In [ ]:
import random
import shutil
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path

import rasterio

LFM_ROOT = Path("/explore/nobackup/projects/lfm/model_inputs/300_300_inputs")
SEM_SEG_ROOT = LFM_ROOT / "kaguya_static_all_wac" / "sem_seg"
SEM_SEG_OUTPUT_ROOT = LFM_ROOT / "full_model_sem_seg_v2"

SEM_SEG_CHIPS_DIR = SEM_SEG_ROOT / "chips"
SEM_SEG_LABELS_DIR = SEM_SEG_ROOT / "labels"

SEM_SEG_IMAGE_GLOB = "*.tif"
SEM_SEG_LABEL_GLOB = "*_label.*"
SEM_SEG_IMAGE_SUFFIX = "_input_wac_static_chip"
SEM_SEG_LABEL_SUFFIX = "_label"

SEM_SEG_SEED = 42
SEM_SEG_N_TEST = 100
SEM_SEG_TRAIN_FRACTION = 0.95
SEM_SEG_MAX_WORKERS = 16


def split_key(path, suffix):
    stem = path.stem
    if suffix and stem.endswith(suffix):
        return stem[: -len(suffix)]
    return stem


def find_sem_seg_pairs():
    chips = {
        split_key(path, SEM_SEG_IMAGE_SUFFIX): path
        for path in sorted(SEM_SEG_CHIPS_DIR.glob(SEM_SEG_IMAGE_GLOB))
    }
    labels = {
        split_key(path, SEM_SEG_LABEL_SUFFIX): path
        for path in sorted(SEM_SEG_LABELS_DIR.glob(SEM_SEG_LABEL_GLOB))
    }

    keys = sorted(set(chips) & set(labels))
    chip_only = sorted(set(chips) - set(labels))
    label_only = sorted(set(labels) - set(chips))

    print(f"matched pairs: {len(keys)}")
    print(f"chips only:    {len(chip_only)}")
    print(f"labels only:   {len(label_only)}")

    if len(keys) <= SEM_SEG_N_TEST:
        raise ValueError(f"Need more than {SEM_SEG_N_TEST} pairs, found {len(keys)}")

    return [(chips[key], labels[key]) for key in keys]


def process_sem_seg_label(label_path: Path, label_out: Path) -> None:
    label_out.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(label_path, label_out)


def write_sem_seg_7band_chip(chip_path: Path, out_path: Path) -> None:
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with rasterio.open(chip_path) as src:
        if src.count < 7:
            raise ValueError(f"{chip_path} has only {src.count} bands")
        profile = src.profile.copy()
        profile.update(driver="GTiff", count=7)
        data = src.read(indexes=list(range(1, 8)))
        with rasterio.open(out_path, "w", **profile) as dst:
            dst.write(data)


def copy_sem_seg_pair_job(args):
    chip_path, label_path, split = args
    chip_out = SEM_SEG_OUTPUT_ROOT / split / "chips" / chip_path.name
    label_out = SEM_SEG_OUTPUT_ROOT / split / "labels" / label_path.name
    write_sem_seg_7band_chip(chip_path, chip_out)
    process_sem_seg_label(label_path, label_out)
    return split


def create_sem_seg_split():
    pairs = find_sem_seg_pairs()
    rng = random.Random(SEM_SEG_SEED)
    rng.shuffle(pairs)

    test_pairs = pairs[:SEM_SEG_N_TEST]
    remaining = pairs[SEM_SEG_N_TEST:]
    n_train = int(round(len(remaining) * SEM_SEG_TRAIN_FRACTION))

    split_pairs = {
        "train": remaining[:n_train],
        "val": remaining[n_train:],
        "test": test_pairs,
    }

    jobs = [
        (chip_path, label_path, split)
        for split, pairs_for_split in split_pairs.items()
        for chip_path, label_path in pairs_for_split
    ]

    with ProcessPoolExecutor(max_workers=SEM_SEG_MAX_WORKERS) as executor:
        list(executor.map(copy_sem_seg_pair_job, jobs))

    for split, pairs_for_split in split_pairs.items():
        print(f"{split}: {len(pairs_for_split)} pairs")
    print(f"wrote split dataset to: {SEM_SEG_OUTPUT_ROOT.resolve()}")

In [ ]:
create_sem_seg_split()

## Create Instance Segmentation Splits

Builds a fresh `train`/`val`/`test` split from the instance-segmentation source directory. Chip TIFFs are saved with only the first 7 bands. `.npz` labels are copied unchanged.

In [ ]:
import random
import shutil
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path

import rasterio

ISEG_SOURCE_ROOT = Path("/explore/nobackup/projects/lfm/model_inputs/300_300_inputs/kaguya_static_all_wac/inst_seg")
ISEG_OUTPUT_ROOT = Path("/explore/nobackup/projects/lfm/model_inputs/300_300_inputs/full_model_inst_seg_v2")

ISEG_CHIPS_DIR = ISEG_SOURCE_ROOT / "chips"
ISEG_LABELS_DIR = ISEG_SOURCE_ROOT / "labels"

ISEG_IMAGE_GLOB = "*.tif"
ISEG_LABEL_GLOB = "*_label.npz"
ISEG_IMAGE_SUFFIX = "_input_wac_static_chip"
ISEG_LABEL_SUFFIX = "_label"

ISEG_SEED = 42
ISEG_N_TEST = 100
ISEG_TRAIN_FRACTION = 0.95
ISEG_MAX_WORKERS = 16


def split_key(path, suffix):
    stem = path.stem
    if suffix and stem.endswith(suffix):
        return stem[: -len(suffix)]
    return stem


def find_iseg_pairs():
    chips = {
        split_key(path, ISEG_IMAGE_SUFFIX): path
        for path in sorted(ISEG_CHIPS_DIR.glob(ISEG_IMAGE_GLOB))
    }
    labels = {
        split_key(path, ISEG_LABEL_SUFFIX): path
        for path in sorted(ISEG_LABELS_DIR.glob(ISEG_LABEL_GLOB))
    }

    keys = sorted(set(chips) & set(labels))
    chip_only = sorted(set(chips) - set(labels))
    label_only = sorted(set(labels) - set(chips))

    print(f"matched pairs: {len(keys)}")
    print(f"chips only:    {len(chip_only)}")
    print(f"labels only:   {len(label_only)}")

    if len(keys) <= ISEG_N_TEST:
        raise ValueError(f"Need more than {ISEG_N_TEST} pairs, found {len(keys)}")

    return [(chips[key], labels[key]) for key in keys]


def write_iseg_7band_chip(chip_path: Path, out_path: Path) -> None:
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with rasterio.open(chip_path) as src:
        if src.count < 7:
            raise ValueError(f"{chip_path} has only {src.count} bands")
        profile = src.profile.copy()
        profile.update(driver="GTiff", count=7)
        data = src.read(indexes=list(range(1, 8)))
        with rasterio.open(out_path, "w", **profile) as dst:
            dst.write(data)


def copy_iseg_pair_job(args):
    chip_path, label_path, split = args
    chip_out = ISEG_OUTPUT_ROOT / split / "chips" / chip_path.name
    label_out = ISEG_OUTPUT_ROOT / split / "labels" / label_path.name
    write_iseg_7band_chip(chip_path, chip_out)
    label_out.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(label_path, label_out)
    return split


def create_iseg_split():
    pairs = find_iseg_pairs()
    rng = random.Random(ISEG_SEED)
    rng.shuffle(pairs)

    test_pairs = pairs[:ISEG_N_TEST]
    remaining = pairs[ISEG_N_TEST:]
    n_train = int(round(len(remaining) * ISEG_TRAIN_FRACTION))

    split_pairs = {
        "train": remaining[:n_train],
        "val": remaining[n_train:],
        "test": test_pairs,
    }

    jobs = [
        (chip_path, label_path, split)
        for split, pairs_for_split in split_pairs.items()
        for chip_path, label_path in pairs_for_split
    ]

    with ProcessPoolExecutor(max_workers=ISEG_MAX_WORKERS) as executor:
        list(executor.map(copy_iseg_pair_job, jobs))

    for split, pairs_for_split in split_pairs.items():
        print(f"{split}: {len(pairs_for_split)} pairs")
    print(f"wrote split dataset to: {ISEG_OUTPUT_ROOT.resolve()}")

In [ ]:
create_iseg_split()

In [ ]:
for split in ["train", "val", "test"]:
    chips = sorted((ISEG_OUTPUT_ROOT / split / "chips").glob("*.tif"))
    labels = sorted((ISEG_OUTPUT_ROOT / split / "labels").glob("*.npz"))
    other_labels = [p for p in (ISEG_OUTPUT_ROOT / split / "labels").iterdir() if p.is_file() and p.suffix != ".npz"]
    print(f"{split}: {len(chips)} chips, {len(labels)} .npz labels, {len(other_labels)} non-npz labels")